In [1]:
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt

c:\Users\knguy139\Documents\Projects\.venv\Lib\site-packages\snowflake\connector\vendored\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [2]:
engine = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

In [3]:
sql_base = """
    select
        d.collection as hospital_group
        , a.fin_market
        , count(distinct a.case_id) as case_count
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt
        , count(distinct case when a.persistentfulladr_cases = 1 then a.case_id end) as persistent_adr_cnt
        , count(distinct case when a.p2p_full_ovtn = 1 then a.case_id end) as p2p_ovrtn_cnt
        , count(distinct case when a.appeal_ovrtn_ind = 1 then a.case_id end) as appeal_ovrtn_cnt
        , count(distinct case when a.mcr_ovtrn_ind = 1 then a.case_id end) as mcr_ovrtn_cnt
        , count(distinct case when a.icm_md_reviewed_ind = 1 then a.case_id end) as md_reviewed_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand = 'M&R'
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and d.collection is not null
        and hce_admit_month between '{start}' and '{end}'
    group by 1,2
    having count(distinct a.case_id) >= 30
"""

df_2025 = pd.read_sql(sql_base.format(start='202501', end='202512'), engine)

df_2026 = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(distinct a.case_id) as case_count
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt
        , count(distinct case when a.persistentfulladr_cases = 1 then a.case_id end) as persistent_adr_cnt
        , count(distinct case when a.p2p_full_ovtn = 1 then a.case_id end) as p2p_ovrtn_cnt
        , count(distinct case when a.appeal_ovrtn_ind = 1 then a.case_id end) as appeal_ovrtn_cnt
        , count(distinct case when a.mcr_ovtrn_ind = 1 then a.case_id end) as mcr_ovrtn_cnt
        , count(distinct case when a.icm_md_reviewed_ind = 1 then a.case_id end) as md_reviewed_cnt
        , count(distinct case when a.member_appeal_ind = 1 then a.case_id end) as member_appeal_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand = 'M&R'
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and d.collection is not null
        and hce_admit_month between '202601' and '202604'
    group by 1,2
    having count(distinct a.case_id) >= 30
""", engine)

print(f"2025: {len(df_2025)} rows | 2026: {len(df_2026)} rows")

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/db05faca-c82a-4b9d-b9c5-0f64b6755421/saml2?SAMLRequest=lVLRbtowFP2VyHtO4pjAwAIqCq1g6lZWaKX2ZXIcB7w6dubrNLRfPyeA1D200h4sWfY595x7zx1fHEoVvAgL0ugJSiKMAqG5yaXeTdD99jocogAc0zlTRosJehWALqZjYKWq6Kx2e30n%2FtQCXOALaaDtxwTVVlPDQALVrBRAHaeb2fcbSiJMGYCwzsuhEyUH6bX2zlU0jpumiZpeZOwuJhjjGI9ij2ohX9A7iepzjcoaZ7hRZ8rB9%2FSBRBLjtJXwCK%2BwPhEvpT6O4DOV7AgCutxu1%2BH6drNFwezc3dxoqEthN8K%2BSC7u726OBsA7qPe70J%2B8YQx%2B7VkE2jSFYs%2BCm7Kqna8Z%2BVtciDxWZif9pFaLCaqeZZ7MqsPuTY%2FgcbNSeHn1pkTyu5%2Fezr9dVo9X6cPm5zxbkuVTT3COgodzrqTNdQVQi5Vu03T%2BCZNBiAch6W0TQntDmg6iHkmeULDwaUrNXMc8W%2B58RKXk1oApnNFKatG5zDPcLxhnIR8SFqbZKA%2BzEe%2BHuBik2eBrv5%2BSJG4zI%2Bi4N7QzYqf%2FN41x%2FJ57WsAfPpPVYm2U5K%2FBtbElcx9HlkRJ9yLzsOigVJRMqlmeWwHgo1PKNHMrmPN77mwtUDw9qv676dO%2F&RelayState=ver%3A3-hint%3A260092117886646-ETMsDgAAAZ70fYExABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEMgFbi5UggBBr49VCaEKhqAAAACQCXgz5v

In [4]:
def mom_optimal_k(successes, trials):
    mask = (trials > 0) & (successes >= 0)
    s, n = successes[mask].values, trials[mask].values
    p = s / n
    p_bar = p.mean()
    v_obs = p.var(ddof=1)
    n_bar = len(n) / np.sum(1.0 / n)
    v_sampling = p_bar * (1 - p_bar) / n_bar
    v_signal = v_obs - v_sampling
    if v_signal <= 0:
        return 50.0
    alpha_hat = p_bar * (p_bar * (1 - p_bar) / v_signal - 1)
    beta_hat = (1 - p_bar) * (p_bar * (1 - p_bar) / v_signal - 1)
    if alpha_hat <= 0 or beta_hat <= 0:
        return 50.0
    return alpha_hat + beta_hat


def compute_shrinkage(df, year, k_map):
    rate_defs = {
        "initial_adr":    ("initial_adr_cnt",    "case_count"),
        "persistent_adr": ("persistent_adr_cnt",  "case_count"),
        "p2p_overturn":   ("p2p_ovrtn_cnt",      "initial_adr_cnt"),
        "appeal_overturn": ("appeal_ovrtn_cnt",   "initial_adr_cnt"),
        "mcr_overturn":   ("mcr_ovrtn_cnt",      "initial_adr_cnt"),
        "md_review":      ("md_reviewed_cnt",     "case_count"),
    }
    if "member_appeal_cnt" in df.columns:
        rate_defs["member_appeal"] = ("member_appeal_cnt", "initial_adr_cnt")

    df_broad = df.query("case_count >= 30").copy()

    global_rates = {}
    for feat, (num, denom) in rate_defs.items():
        global_rates[feat] = df_broad[num].sum() / df_broad[denom].replace(0, np.nan).sum()

    df_out = df.query("case_count >= 30 and initial_adr_cnt >= 10").copy()
    df_out["year"] = year

    for feat, (num, denom) in rate_defs.items():
        k = k_map.get(feat, 50.0)
        gr = global_rates[feat]
        df_out[f"{feat}_adj_rate"] = (df_out[num] + k * gr) / (df_out[denom] + k)

    df_out["log_case_count"] = np.log1p(df_out["case_count"])
    df_out["log_initial_adr_cnt"] = np.log1p(df_out["initial_adr_cnt"])

    rate_cols = [c for c in df_out.columns if c.endswith("_adj_rate")]
    df_out[rate_cols] = df_out[rate_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

    return df_out

In [5]:
df_all = pd.concat([df_2025, df_2026], ignore_index=True).query("case_count >= 30")

rate_configs = {
    "initial_adr":    ("initial_adr_cnt",    "case_count"),
    "persistent_adr": ("persistent_adr_cnt",  "case_count"),
    "p2p_overturn":   ("p2p_ovrtn_cnt",      "initial_adr_cnt"),
    "appeal_overturn": ("appeal_ovrtn_cnt",   "initial_adr_cnt"),
    "mcr_overturn":   ("mcr_ovrtn_cnt",      "initial_adr_cnt"),
    "md_review":      ("md_reviewed_cnt",     "case_count"),
}

k_map = {}
for feat, (num, denom) in rate_configs.items():
    k_map[feat] = mom_optimal_k(df_all[num], df_all[denom])

k_map["member_appeal"] = mom_optimal_k(
    df_2026.query("case_count >= 30")["member_appeal_cnt"],
    df_2026.query("case_count >= 30")["initial_adr_cnt"]
)

print("k_map:")
for k, v in k_map.items():
    print(f"  {k:<20}: {v:.1f}")

k_map:
  initial_adr         : 8.3
  persistent_adr      : 15.1
  p2p_overturn        : 8.8
  appeal_overturn     : 1.1
  mcr_overturn        : 204.8
  md_review           : 2.0
  member_appeal       : 5.0


In [6]:
iso_kpi_9 = [
    "log_case_count", "log_initial_adr_cnt",
    "initial_adr_adj_rate", "persistent_adr_adj_rate", "p2p_overturn_adj_rate",
    "appeal_overturn_adj_rate", "mcr_overturn_adj_rate", "md_review_adj_rate",
    "member_appeal_adj_rate"
]

iso_kpi_8 = [c for c in iso_kpi_9 if c != "member_appeal_adj_rate"]


def run_if_and_flag(X, percentile=2.5, random_state=42):
    model = IsolationForest(contamination="auto", n_estimators=300, random_state=random_state)
    model.fit(X)
    scores = model.decision_function(X)
    threshold = np.percentile(scores, percentile)
    return scores < threshold, scores, threshold


def extract_flagged(df, flag_col):
    return set(df.loc[df[flag_col], ["hospital_group", "fin_market"]].apply(tuple, axis=1))

In [7]:
df_iso_25 = compute_shrinkage(df_2025.copy(), "2025", k_map)
df_iso_26 = compute_shrinkage(df_2026.copy(), "2026", k_map)

gr_ma_26 = df_iso_26["member_appeal_adj_rate"].mean()
df_iso_25["member_appeal_adj_rate"] = gr_ma_26

df_pooled = pd.concat([df_iso_25, df_iso_26], ignore_index=True)
df_pooled[iso_kpi_9] = df_pooled[iso_kpi_9].replace([np.inf, -np.inf], np.nan).fillna(0)

flags_base, _, _ = run_if_and_flag(df_pooled[iso_kpi_9].values)
df_pooled["flag_baseline"] = flags_base

flagged_25_base = extract_flagged(df_pooled.query("year == '2025'"), "flag_baseline")
flagged_26_base = extract_flagged(df_pooled.query("year == '2026'"), "flag_baseline")

print(f"BASELINE (9 feat, imputed): 2025={len(flagged_25_base)}, 2026={len(flagged_26_base)}")

BASELINE (9 feat, imputed): 2025=35, 2026=16


In [8]:
df_iso_25_opt1 = compute_shrinkage(df_2025.copy(), "2025", k_map)
df_iso_26_opt1 = compute_shrinkage(df_2026.copy(), "2026", k_map)

df_pooled_opt1 = pd.concat([df_iso_25_opt1, df_iso_26_opt1], ignore_index=True)
df_pooled_opt1[iso_kpi_8] = df_pooled_opt1[iso_kpi_8].replace([np.inf, -np.inf], np.nan).fillna(0)

flags_opt1, _, _ = run_if_and_flag(df_pooled_opt1[iso_kpi_8].values)
df_pooled_opt1["flag_opt1"] = flags_opt1

flagged_25_opt1 = extract_flagged(df_pooled_opt1.query("year == '2025'"), "flag_opt1")
flagged_26_opt1 = extract_flagged(df_pooled_opt1.query("year == '2026'"), "flag_opt1")

print(f"OPTION 1 (8 feat, no member_appeal): 2025={len(flagged_25_opt1)}, 2026={len(flagged_26_opt1)}")

OPTION 1 (8 feat, no member_appeal): 2025=43, 2026=8


In [9]:
df_iso_25_opt3 = compute_shrinkage(df_2025.copy(), "2025", k_map)
df_iso_25_opt3[iso_kpi_8] = df_iso_25_opt3[iso_kpi_8].replace([np.inf, -np.inf], np.nan).fillna(0)
flags_25_opt3, _, _ = run_if_and_flag(df_iso_25_opt3[iso_kpi_8].values)
df_iso_25_opt3["flag_opt3"] = flags_25_opt3
flagged_25_opt3 = extract_flagged(df_iso_25_opt3, "flag_opt3")

df_iso_26_opt3 = compute_shrinkage(df_2026.copy(), "2026", k_map)
df_iso_26_opt3[iso_kpi_9] = df_iso_26_opt3[iso_kpi_9].replace([np.inf, -np.inf], np.nan).fillna(0)
flags_26_opt3, _, _ = run_if_and_flag(df_iso_26_opt3[iso_kpi_9].values)
df_iso_26_opt3["flag_opt3"] = flags_26_opt3
flagged_26_opt3 = extract_flagged(df_iso_26_opt3, "flag_opt3")

print(f"OPTION 3 (separate models): 2025={len(flagged_25_opt3)} (8feat), 2026={len(flagged_26_opt3)} (9feat)")

OPTION 3 (separate models): 2025=31 (8feat), 2026=20 (9feat)


In [10]:
def jaccard(a, b):
    if len(a) == 0 and len(b) == 0:
        return 1.0
    return len(a & b) / len(a | b)

results = []
for label, f25, f26 in [
    ("Baseline (9feat imputed)", flagged_25_base, flagged_26_base),
    ("Opt1 (8feat pooled)",      flagged_25_opt1, flagged_26_opt1),
    ("Opt3 (separate models)",   flagged_25_opt3, flagged_26_opt3),
]:
    results.append({
        "option": label,
        "flagged_25": len(f25),
        "flagged_26": len(f26),
        "jaccard_25_vs_base": jaccard(flagged_25_base, f25),
        "jaccard_26_vs_base": jaccard(flagged_26_base, f26),
        "gained_25": len(f25 - flagged_25_base),
        "lost_25": len(flagged_25_base - f25),
        "gained_26": len(f26 - flagged_26_base),
        "lost_26": len(flagged_26_base - f26),
    })

df_compare = pd.DataFrame(results).set_index("option")
print(df_compare.to_string())

                          flagged_25  flagged_26  jaccard_25_vs_base  jaccard_26_vs_base  gained_25  lost_25  gained_26  lost_26
option                                                                                                                          
Baseline (9feat imputed)          35          16            1.000000            1.000000          0        0          0        0
Opt1 (8feat pooled)               43           8            0.813953            0.411765          8        0          1        9
Opt3 (separate models)            31          20            0.692308            0.636364          4        8          6        2


In [11]:
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"""
Baseline (v4 current): 9 feat, member_appeal imputed for 2025
  -> 2025: {len(flagged_25_base)} flagged | 2026: {len(flagged_26_base)} flagged

Option 1 (drop member_appeal): 8 feat pooled
  -> 2025: {len(flagged_25_opt1)} flagged | 2026: {len(flagged_26_opt1)} flagged
  -> Jaccard vs baseline: 2025={jaccard(flagged_25_base, flagged_25_opt1):.3f}, 2026={jaccard(flagged_26_base, flagged_26_opt1):.3f}

Option 3 (separate models): 2025=8feat solo, 2026=9feat solo
  -> 2025: {len(flagged_25_opt3)} flagged | 2026: {len(flagged_26_opt3)} flagged
  -> Jaccard vs baseline: 2025={jaccard(flagged_25_base, flagged_25_opt3):.3f}, 2026={jaccard(flagged_26_base, flagged_26_opt3):.3f}

NOTE: If Jaccard > 0.85, options produce near-identical flags and the
choice is low-stakes. Run multi-seed stability in production v5 to confirm.
""")


SUMMARY

Baseline (v4 current): 9 feat, member_appeal imputed for 2025
  -> 2025: 35 flagged | 2026: 16 flagged

Option 1 (drop member_appeal): 8 feat pooled
  -> 2025: 43 flagged | 2026: 8 flagged
  -> Jaccard vs baseline: 2025=0.814, 2026=0.412

Option 3 (separate models): 2025=8feat solo, 2026=9feat solo
  -> 2025: 31 flagged | 2026: 20 flagged
  -> Jaccard vs baseline: 2025=0.692, 2026=0.636

NOTE: If Jaccard > 0.85, options produce near-identical flags and the
choice is low-stakes. Run multi-seed stability in production v5 to confirm.



In [12]:
seeds = list(range(10))

def run_multiseed(X, seeds, percentile=2.5):
    flag_sets = []
    for s in seeds:
        flags, _, _ = run_if_and_flag(X, percentile=percentile, random_state=s)
        flag_sets.append(set(np.where(flags)[0]))
    return flag_sets

def pairwise_jaccard_mean(flag_sets):
    jaccards = []
    for i in range(len(flag_sets)):
        for j in range(i + 1, len(flag_sets)):
            a, b = flag_sets[i], flag_sets[j]
            jaccards.append(len(a & b) / len(a | b) if len(a | b) > 0 else 1.0)
    return np.mean(jaccards), np.std(jaccards)

print("Multi-seed stability (10 seeds, pairwise Jaccard)")
print("=" * 55)

base_25_sets = run_multiseed(df_pooled.query("year == '2025'")[iso_kpi_9].values, seeds)
base_26_sets = run_multiseed(df_pooled.query("year == '2026'")[iso_kpi_9].values, seeds)
base_pooled_sets = run_multiseed(df_pooled[iso_kpi_9].values, seeds)

opt3_25_sets = run_multiseed(df_iso_25_opt3[iso_kpi_8].values, seeds)
opt3_26_sets = run_multiseed(df_iso_26_opt3[iso_kpi_9].values, seeds)

print(f"\nBASELINE (pooled 9feat):")
m, s = pairwise_jaccard_mean(base_pooled_sets)
print(f"  Pooled:  Jaccard = {m:.3f} ± {s:.3f}")
m, s = pairwise_jaccard_mean(base_25_sets)
print(f"  2025 slice: Jaccard = {m:.3f} ± {s:.3f}")
m, s = pairwise_jaccard_mean(base_26_sets)
print(f"  2026 slice: Jaccard = {m:.3f} ± {s:.3f}")

print(f"\nOPTION 3 (separate models):")
m, s = pairwise_jaccard_mean(opt3_25_sets)
print(f"  2025 (8feat): Jaccard = {m:.3f} ± {s:.3f}")
m, s = pairwise_jaccard_mean(opt3_26_sets)
print(f"  2026 (9feat): Jaccard = {m:.3f} ± {s:.3f}")

print("\n" + "-" * 55)
print("Interpretation:")
print("  > 0.85 = stable (seed choice doesn't matter)")
print("  0.70-0.85 = moderate (some noise in flags)")
print("  < 0.70 = unstable (results not trustworthy)")

Multi-seed stability (10 seeds, pairwise Jaccard)

BASELINE (pooled 9feat):
  Pooled:  Jaccard = 0.764 ± 0.064
  2025 slice: Jaccard = 0.754 ± 0.091
  2026 slice: Jaccard = 0.862 ± 0.065

OPTION 3 (separate models):
  2025 (8feat): Jaccard = 0.737 ± 0.071
  2026 (9feat): Jaccard = 0.862 ± 0.065

-------------------------------------------------------
Interpretation:
  > 0.85 = stable (seed choice doesn't matter)
  0.70-0.85 = moderate (some noise in flags)
  < 0.70 = unstable (results not trustworthy)
